This notebook includes linguistic analysis to support the results presented in the report:

  - TF-IDF feature coefficients to identfiy lexical features the model identifies
  - analysis of vocabularly diversity between change and hold meetings
  - a specific analysis of novel vocabulary from false negative meetings in test set

In [1]:
# mount google drive and import all relevant packages
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

Mounted at /content/drive


In [2]:
# import central_bank_minutes_labeled.csv and create meeting_date and year columns to assist analysis
df = pd.read_csv('/content/drive/MyDrive/central_bank_minutes_labeled.csv')
df['meeting_date'] = pd.to_datetime(df['meeting_date'])
df['year'] = df['meeting_date'].dt.year

In [3]:
# use regex to remove text with potential data leakage and/or temporal memorization
def remove_years(text):
    return re.sub(r'\b(19|20)\d{2}\b', '', str(text))

# limit to FOMC data and create train and test sets for TF-IDF coefficient analysis
fomc = df[df['bank'] == 'FOMC'].copy()
train = fomc[fomc['year'] <= 2019].reset_index(drop = True)
test  = fomc[fomc['year'] >= 2020].reset_index(drop = True)

In [5]:
# TF-IDF coefficient analysis (repeating from modeling_and_visualization notebook)

# initialize TF-IDF, remove years from text, and fit and transform
vec = TfidfVectorizer(max_features = 10000  # limited to top 10,000 most frequent terms
                      , ngram_range = (1,2) # include both inslge and two word phrases
                      , min_df = 2)         # word must appear in at least 2 docuemnts to be included
X_train = vec.fit_transform(train['text'].apply(remove_years))

# train and fit with logistic regression classifier
clf  = LogisticRegression(class_weight = 'balanced' # manages class imbalance
                          , max_iter = 1000         # default 100, 10X more iterations added
                          , C = 1.0)                # default L2 regularization to manage overfitting
clf.fit(X_train, train['label'])

# get vocabulary list and coefficients from TF-IDF
feature_names = vec.get_feature_names_out()
coefs = clf.coef_[0]

# sort coefficients in ascending order and get strongest predictors for change and hold respectively
top_change = np.argsort(coefs)[-15:][::-1]
top_hold   = np.argsort(coefs)[:15]

# loop through and print top scoring vocabulary for change and hold
print('Top Scoring Change Vocab:')
for i in top_change:
    print(f'  {feature_names[i]:30s}  {coefs[i]:+.4f}')

print('Top Scoring Hold Vocab:')
for i in top_hold:
    print(f'  {feature_names[i]:30s}  {coefs[i]:+.4f}')

Top Scoring Change Vocab:
  the                             +0.6326
  securities                      +0.3267
  purchases                       +0.3058
  the committee                   +0.2917
  asset                           +0.2718
  committee                       +0.2448
  longer                          +0.2281
  recovery                        +0.2123
  program                         +0.2073
  participants                    +0.2063
  mbs                             +0.1955
  loans                           +0.1952
  continued                       +0.1882
  mandate                         +0.1872
  its                             +0.1841
Top Scoring Hold Vocab:
  mr                              -0.5491
  had                             -0.3495
  in                              -0.3383
  hurricane                       -0.3092
  messrs                          -0.3005
  ms                              -0.2981
  easing                          -0.2530
  basis                   

In [8]:
# calculate type-token ratio (TTR)
def ttr(text, n = 1000):

    # convert text to lowercase and split individual word tokens
    tokens = str(text).lower().split()[:n]

    # divide unique word count by total word count and return result
    return len(set(tokens)) / len(tokens) if tokens else 0 # handle divide by zero errors

# apply ttr() function to fomc data, calculate mean and standard devication, and print result
fomc['ttr'] = fomc['text'].apply(ttr)
print('Type-Token Ratio (TTR) for change and hold labels:')
print(fomc.groupby('label')['ttr'].agg(['mean', 'std']).round(3))

Type-Token Ratio (TTR) for change and hold labels:
         mean    std
label               
change  0.444  0.032
hold    0.441  0.038


Vocabularly use across change and hold is lexically and statistically identical as confirmed by TTR results above. Explains why TF-IDF fails to surpass performance of the baseline majority class model.

In [10]:
# phrase frequency analysis of 3 false negative FOMC meetings from ModernBERT in test set
# compares vocabulary frequency in missed versus correctly labeled changes

fn_dates = ['2022-09', '2023-02', '2024-09'] # false negatives from FOMC test data
correct_dates = ['2022-03', '2022-05', '2022-06', '2022-07', '2022-11', '2022-12'] # correct change dates from FOMC test data

# filter to false negative meetings and correct dates by matching year-month strings
fn_docs = test[test['meeting_date'].dt.strftime('%Y-%m').isin(fn_dates)]
cc_docs = test[test['meeting_date'].dt.strftime('%Y-%m').isin(correct_dates)]

# pull all hold meetings for comparison
hold_docs = test[test['label'] == 'hold']

# list out candidate phrases found in each false negative meeting
phrases = [
    'disinflation'
    , 'restrictive'
    , 'cumulative'
    , 'sufficiently'
    , 'basis points'
    , 'appropriate'
    , 'inflation'
    , 'labor market'
    , 'uncertain',
]

In [15]:
# define function to calculate frequency of word usage per 1,000 words
def freq_per_1000(docs, phrase):

    # get total word count for denominator with string split
    total_words = docs['text'].str.split().str.len().sum()

    # calculate how frequently each candidate appears in all documents
    total_count = docs['text'].str.lower().str.count(re.escape(phrase)).sum()

    # return frequency per 1,000
    return round((total_count / total_words) * 1000, 3) if total_words > 0 else 0 # divide by zero handling

# loop through each candidate word and compute frequency per 1,000, return result
rows = []
for i in phrases:
    rows.append({'phrase': i
        , 'false_neg': freq_per_1000(fn_docs, i)
        , 'correct_chg': freq_per_1000(cc_docs, i)
        , 'hold': freq_per_1000(hold_docs, i)
    })

# convert to dataframe and print
print(pd.DataFrame(rows))

         phrase  false_neg  correct_chg   hold
0  disinflation      0.129        0.000  0.153
1   restrictive      1.251        0.720  0.320
2    cumulative      0.474        0.206  0.073
3  sufficiently      0.345        0.350  0.070
4  basis points      0.302        0.453  0.237
5   appropriate      1.898        2.469  1.786
6     inflation     10.912       11.213  8.192
7  labor market      2.243        1.728  1.807
8     uncertain      0.906        1.214  1.278
